In [ ]:
# Microsoft Partner Center Ingestion

Ingests customer enrichment data from the **Microsoft Partner Center REST API** into the `ManagedServiceData` Lakehouse,
supplementing the raw Inforcer assessment tables with CSP-level tenant detail.

## Tables Produced (prefix `pc_`)

| Table | Grain | Represents |
|-------|-------|------------|
| `pc_customers` | one row per CSP customer | Customer profile, tenant ID, domain, relationship status |
| `pc_subscriptions` | one row per subscription | License SKU, start/end dates, quantity, auto-renew, status |
| `pc_subscribed_skus` | one row per SKU × customer | Available license units, consumed, capability status |
| `pc_promotions` | one row per active promotion | Active NCE promotions, eligible segments, discount %, dates |
| `pc_mci_engagements` | one row per customer × engagement | MCI workshop / engagement eligibility (Eligible / Ineligible / Complete) |

In [3]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# ── Reliance partner tenant ──────────────────────────────────────────────────
TENANT_ID  = "0b60fed4-5fc9-409d-95f2-271114f4c86f"   # relianceinfosystems.com
CLIENT_ID  = "598e341b-9177-46fd-a104-df705cb8e036"   # POA MCP Server (Partner Center App Management)

# ── Key Vault ────────────────────────────────────────────────────────────────
KEY_VAULT_URL  = "https://dynamicsfabricsynckey.vault.azure.net/"
PC_SECRET_NAME = "Partnercenterxinforcerkey"           # App client secret

# ── Incentives — App+User (Secure Application Model) ────────────────────────
# Incentives APIs require a delegated token for a user with "Incentive admin"
# role. Run the "One-Time Consent Setup" cell once to generate this token and
# store it in Key Vault under the name below.
PC_INCENTIVE_REFRESH_SECRET = "PCIncentiveRefreshToken"  # Key Vault secret name

# ── Partner Center API ───────────────────────────────────────────────────────
PC_BASE_URL = "https://api.partnercenter.microsoft.com"
PC_RESOURCE = "https://api.partnercenter.microsoft.com"

# ── Promotion filter ─────────────────────────────────────────────────────────
PROMO_COUNTRY = "NG"         # Two-letter ISO country code (Nigeria)
PROMO_SEGMENT = "commercial" # commercial | education | government

# ── Target Lakehouse ─────────────────────────────────────────────────────────
SCHEMA = "dbo"               # ManagedServiceData uses schema-enabled Lakehouse

# ── Pagination ───────────────────────────────────────────────────────────────
PAGE_SIZE = 300

print("Configuration loaded.")
print(f"  Tenant : {TENANT_ID}")
print(f"  Client : {CLIENT_ID}")
print(f"  KV     : {KEY_VAULT_URL} / {PC_SECRET_NAME}")
print(f"  PC URL : {PC_BASE_URL}")
print(f"  Promo  : country={PROMO_COUNTRY}, segment={PROMO_SEGMENT}")
print(f"  Schema : {SCHEMA}")

StatementMeta(, de006c7b-faf7-4d54-acbc-f7566fd78903, 15, Finished, Available, Finished, False)

Configuration loaded.
  Tenant : 0b60fed4-5fc9-409d-95f2-271114f4c86f
  Client : 598e341b-9177-46fd-a104-df705cb8e036
  KV     : https://dynamicsfabricsynckey.vault.azure.net/ / Partnercenterxinforcerkey
  PC URL : https://api.partnercenter.microsoft.com
  Promo  : country=NG, segment=commercial
  Schema : dbo


In [ ]:
# ============================================================================
# INSTALL DEPENDENCIES
# ============================================================================
%pip install msal --quiet

StatementMeta(, de006c7b-faf7-4d54-acbc-f7566fd78903, 13, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [4]:
# ============================================================================
# AUTHENTICATE — Partner Center API helpers
# ============================================================================
# Two token strategies:
#   1. App-only (client credentials)  — used for customer/subscription/promo APIs
#   2. App+User (Secure App Model)    — used for Incentives APIs only
#      Requires a refresh token (for an Incentive admin user) in Key Vault.
#      Run the "One-Time Consent Setup" cell to generate it.

import msal
import requests
import json
import time
from datetime import datetime, timezone

print("Retrieving Partner Center client secret from Key Vault...")
client_secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, PC_SECRET_NAME)

_msal_app = msal.ConfidentialClientApplication(
    client_id=CLIENT_ID,
    client_credential=client_secret,
    authority=f"https://login.microsoftonline.com/{TENANT_ID}",
)

# ── 1. App-only token (customers / subscriptions / promotions) ───────────────
_token_cache = {"token": None, "expires_at": 0}

def get_pc_token():
    """Return a valid app-only Bearer token, refreshing if needed."""
    if _token_cache["token"] and time.time() < _token_cache["expires_at"] - 60:
        return _token_cache["token"]
    result = _msal_app.acquire_token_for_client(scopes=[f"{PC_RESOURCE}/.default"])
    if "access_token" not in result:
        raise RuntimeError(f"Token acquisition failed: {result.get('error')} — {result.get('error_description')}")
    _token_cache["token"] = result["access_token"]
    _token_cache["expires_at"] = time.time() + result.get("expires_in", 3600)
    return _token_cache["token"]

# ── 2. Delegated token (Incentives APIs — requires Incentive admin user) ─────
_inc_token_cache = {"token": None, "expires_at": 0, "refresh_token": None}

def get_pc_incentive_token():
    """Return a delegated Bearer token for Incentives APIs.
    Uses a refresh token stored in Key Vault (Secure Application Model).
    Run the 'One-Time Consent Setup' cell first if this fails.
    """
    if _inc_token_cache["token"] and time.time() < _inc_token_cache["expires_at"] - 60:
        return _inc_token_cache["token"]
    # Load refresh token from Key Vault on first use
    if _inc_token_cache["refresh_token"] is None:
        _inc_token_cache["refresh_token"] = notebookutils.credentials.getSecret(
            KEY_VAULT_URL, PC_INCENTIVE_REFRESH_SECRET
        )
    result = _msal_app.acquire_token_by_refresh_token(
        _inc_token_cache["refresh_token"],
        scopes=["https://api.partnercenter.microsoft.com/user_impersonation"]
    )
    if "access_token" not in result:
        raise RuntimeError(
            f"Incentive token refresh failed: {result.get('error')} — {result.get('error_description')}\n"
            "Re-run the 'One-Time Consent Setup' cell to generate a fresh refresh token."
        )
    _inc_token_cache["token"] = result["access_token"]
    _inc_token_cache["expires_at"] = time.time() + result.get("expires_in", 3600)
    if "refresh_token" in result:
        _inc_token_cache["refresh_token"] = result["refresh_token"]
    return _inc_token_cache["token"]

# ── Generic GET helpers ───────────────────────────────────────────────────────
def _pc_get_with_token(token_fn, path, params=None):
    headers = {
        "Authorization": f"Bearer {token_fn()}",
        "Accept": "application/json",
        "MS-Contract-Version": "v1",
        "X-Locale": "en-US",
    }
    url = f"{PC_BASE_URL}/{path.lstrip('/')}"
    resp = requests.get(url, headers=headers, params=params, timeout=60)
    if resp.status_code == 429:
        time.sleep(int(resp.headers.get("Retry-After", 10)))
        resp = requests.get(url, headers=headers, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

def pc_get(path, params=None):
    """App-only GET — customers, subscriptions, promotions."""
    return _pc_get_with_token(get_pc_token, path, params)

def pc_get_incentive(path, params=None):
    """Delegated GET — Incentives / MCI endpoints (requires Incentive admin refresh token)."""
    return _pc_get_with_token(get_pc_incentive_token, path, params)

def pc_get_paged(path, params=None):
    """Offset-based pagination for app-only endpoints."""
    params = dict(params or {})
    params["size"] = PAGE_SIZE
    offset = 0
    total = None
    while True:
        data = pc_get(path, {**params, "offset": offset})
        items = data.get("items", [])
        if not items:
            break
        for item in items:
            yield item
        if total is None:
            total = data.get("totalCount", 0)
        offset += len(items)
        if offset >= total:
            break

_ = get_pc_token()
print("Successfully authenticated to Partner Center API (app-only).")
print("Incentive token will load on first use of pc_get_incentive() — requires Key Vault secret.")
print(f"  Incentive refresh secret name: {PC_INCENTIVE_REFRESH_SECRET}")

StatementMeta(, de006c7b-faf7-4d54-acbc-f7566fd78903, 16, Finished, Available, Finished, False)

Retrieving Partner Center client secret from Key Vault...
Successfully authenticated to Partner Center API (app-only).
Incentive token will load on first use of pc_get_incentive() — requires Key Vault secret.
  Incentive refresh secret name: PCIncentiveRefreshToken


In [20]:
# ============================================================================
# ONE-TIME CONSENT SETUP — Generate Incentive admin refresh token
# ============================================================================
# STEP 1 — Run this cell with RUN_CONSENT_SETUP = True to print the auth URL.
# STEP 2 — Open the URL, sign in as Incentive admin, copy the redirect URL
#           from the browser address bar (starts with https://localhost/?code=...)
# STEP 3 — Paste that full redirect URL into REDIRECT_RESPONSE_URL below,
#           then re-run this cell. The refresh token is saved to Key Vault.
# STEP 4 — Set RUN_CONSENT_SETUP = False and never run this cell again.

RUN_CONSENT_SETUP    = False    # ← keep True until token is saved
REDIRECT_RESPONSE_URL = "https://localhost/?code=1.AQkA1P5gC8lfnUCV8icRFPTIbxs0jll3kf1GoQTfcFy44DYAAOAJAA.BQABBAIAAAADAOz_BQD0_0V2b1N0c0FydGlmYWN0cwIAAAAAAP57F3e6cIB4tp-jS45gIIWV9w6ReQzAS3o5wQJB25Ozx3YgpjFO-40gKmWpdcCgwTFfcUSNxMkuR5d8p_BxzacwWWcSdVT8TCM5TIzGZW3CJla4c7cS5KPzxX-Q1DklAKGDqaqjQekEh5SoJqaVvJH_yJL6SpRwFPUiIYiCgdaK0lDV02VoC08HOMWmh1IjWIVVKGBJMFPgAbKumoOhv3sJOHP5P11erV8agwFqKlpFITGJ9ueZ2mdiPSiPxQYXBZsr8QFmxV7dwHHfb7gv47MV3YUIcXeihQW4dMwnI1C0nSTYYlEBT9C4egTa-qVuI6j9H4WfCjnJYMp0hMGCt4aoha4LYDjhKinX-eJGz5fiROa6-WmV4c8GPLk3cyiLpmhIV-5mad5QD_NirlvNHCdDvP9Q-VqwaveVdUqipVO9xSqwloWRUDTb0vZuAbIkdDp-ExtIByt_uKFgwOmEqiYOadENcIw_GpFg-MxTuhmxWpHrrBuUXTVbUtvn76lM6fxqesnXgg4CJp6kskK6BNqERhqsWpC4ywUOkUUn-N5Ut5kbkz3GB-E_PevytT4mxptMX0ykTXJmSoLzm9tLi2lNUWVNraSImZ-OxDHuzW-XKcCOdWOc3nvWjLur5OkWB_wMLFKlx4okOjGrN9P6qibidsU7ozRONNQWLe_9_MKVwdZe85K1idT8X11rh2Q8A56nC2Ln1Z48UON-PM-chHpELZGFmJizXbuVgBigbjIa5tFicAjos2CZj6AvaeYnf1bfThg67zJwmFCffVL037Xx2hhmChwDaS2BI7fZIdczSr7dLIoOTp2oeF_oT7Cs9otgYvf0GdIjnUkUdZm0CWKRCKx8Nks_AtLkpYOcc7Mkz2tcue54_dXmdMqvP65ULbc5DEKSoxHqyx6zu6cuIv-f3cpZcZ5-_75cb4du9DDvxLM9Z1bj_al-6cRg9CVuVN3ZSADaYq7nrkdiVzBVimb9r28LUtWzE-AbCTDQpZ5BWuusvw8XBvXGkSg5d_rPbuqhphkHUvA8SldBCS0e3n_XGg1RQQWK_MV-2VoJ3LBhpm65IgLUojCpCSeYHW-Stc9ULxjC_upuZlmQ8TiuASbovCDbJhVTat3IgraGriHvolfvMI2SIj35GgbBYICjZUxdgdESCvC6pEuZrJNjliTpwa3R39mIGZntQsMi0z4d0HjSq3TZYdFWHDPAol2yQDX4iWm0CTDDzvTiRtBxXM3JODed_RBfEUmR-W77lNlUq1tnuok68jCr1sIo6YSflYvBflb4Xw_EIvjgxBWthwGfNMe1CeTONU9l8kgmkVvd0Q&session_state=006c5e4a-2ec6-2e1e-8de2-891eaf15bfe5"     # ← paste the full https://localhost/?code=... URL here after Step 2

# ─────────────────────────────────────────────────────────────────────────────
if not RUN_CONSENT_SETUP:
    print("Consent setup skipped (RUN_CONSENT_SETUP = False).")
else:
    REDIRECT_URI    = "https://localhost"
    INCENTIVE_SCOPE = ["https://api.partnercenter.microsoft.com/user_impersonation"]

    if not REDIRECT_RESPONSE_URL:
        # ── Phase 1: print the auth URL for the user to open ─────────────────
        auth_url = _msal_app.get_authorization_request_url(
            scopes=INCENTIVE_SCOPE,
            redirect_uri=REDIRECT_URI,
            prompt="consent",
        )
        print("=" * 70)
        print("ACTION: Open the URL below in a browser and sign in as your Incentive admin user.")
        print("=" * 70)
        print()
        print(auth_url)
        print()
        print("After sign-in the browser will show 'This site can't be reached'.")
        print("Copy the FULL URL from the address bar (starts with https://localhost/?code=...).")
        print()
        print("Then paste it into REDIRECT_RESPONSE_URL at the top of this cell and re-run.")
    else:
        # ── Phase 2: exchange the code for tokens ─────────────────────────────
        from urllib.parse import urlparse, parse_qs
        parsed = urlparse(REDIRECT_RESPONSE_URL.strip())
        code   = parse_qs(parsed.query).get("code", [None])[0]

        if not code:
            print("ERROR: could not extract 'code' from REDIRECT_RESPONSE_URL.")
            print("Make sure you pasted the full redirect URL including the ?code=... part.")
        else:
            print("Exchanging auth code for tokens...")
            result = _msal_app.acquire_token_by_authorization_code(
                code,
                scopes=INCENTIVE_SCOPE,
                redirect_uri=REDIRECT_URI,
            )
            if "refresh_token" not in result:
                print(f"ERROR: {result.get('error')}: {result.get('error_description')}")
                print("The auth code may have expired (valid ~10 min). Re-run Phase 1 to get a new URL.")
            else:
                refresh_token = result["refresh_token"]
                print("✓ Token exchange successful.")

                # ── Save to Key Vault (uses notebookutils Fabric identity) ────────
                try:
                    _kv_token = notebookutils.credentials.getToken("https://vault.azure.net")
                    _kv_url   = f"{KEY_VAULT_URL.rstrip('/')}/secrets/{PC_INCENTIVE_REFRESH_SECRET}?api-version=7.4"
                    _kv_resp  = requests.put(
                        _kv_url,
                        headers={"Authorization": f"Bearer {_kv_token}", "Content-Type": "application/json"},
                        json={"value": refresh_token},
                        timeout=30,
                    )
                    _kv_resp.raise_for_status()
                    print(f"✓ Refresh token saved to Key Vault as '{PC_INCENTIVE_REFRESH_SECRET}'")
                    print()
                    print("DONE — set RUN_CONSENT_SETUP = False and run Step 5 (MCI cell).")
                except Exception as e:
                    print(f"Could not auto-save to Key Vault: {e}")
                    print()
                    print("Manually add this secret in the Azure portal:")
                    print(f"  Vault  : {KEY_VAULT_URL}")
                    print(f"  Name   : {PC_INCENTIVE_REFRESH_SECRET}")
                    print(f"  Value  : (shown below — copy the entire string)")
                    print()
                    print(refresh_token)

StatementMeta(, 929c1108-0cb1-4e64-8d97-e2f9bfc54718, 30, Finished, Available, Finished, False)

Consent setup skipped (RUN_CONSENT_SETUP = False).


In [21]:
# ============================================================================
# ONE-TIME: Save refresh token to Key Vault
# ============================================================================
# Run this cell once to save the refresh token that was printed by cell 5.
# Paste the token value from the output above into REFRESH_TOKEN_VALUE.
# Delete or skip this cell after the secret is confirmed in Key Vault.

REFRESH_TOKEN_VALUE = "1.AQkA1P5gC8lfnUCV8icRFPTIbxs0jll3kf1GoQTfcFy44DYAAOAJAA.BQABAwEAAAADAOz_BQD0_0V2b1N0c0FydGlmYWN0cwIAAAAAACTbn_Tx1zpp6Kj9w11PQ6cMUTX0t1VUyf7UHo_7VB9N2kXKRB7KElOtKzhi4FaIJPMtcZkjTpRR2uckRBd5wH62TJiD-R9EfTX2lRcyaHXFG3iuYlECWqb6MKMMIcHaa1fD_hj5z5692_DAsPqS-x8ilX6sQJOEuNK1VG4yn1e6-_51o9kDXDXj4oZ36lN9lqVW-fnDZMi9Xf4-QkqbiqEkDAXaUi7YH3o9JnIzoMXacLqFxFrg01VT218Cx00_blMrFQsYATrN_yCHGwfWPlSVYU2_TH1NVfGTgb_O42Z-Xe_BieLUVz1PNc75Iv_5JW3FMqh-KQQXXgwJuBKWBRejGS4C3KeXDkzTQFZOZ3oqGNZqBEXsNcK-1lPP3z-ae_nV0aRE6gj4NCm4yoRII0ttxqXXlcAJ2nfrhYdascddA_yFiih0NCUJedoVNp0TNj1iumgReXQVDEJloEDpkUCeRQvhDeK8h6VvWdwWrFFCSchx4FmHtVt7NseIoBZxfSQzGw-Xbu1tlPBCOuYGZd1r0y1t7S0Zq1oOihuC0FRSHcGJ0eBJVEbUmZllgcsYEtaCTCgW_Jx-6W-cS348Bfdkdz2fZDKkbJXC07kwlRIOuskl-CJTTz2eWC-AZh5fSmDrvKAcEU4fV9ACBjTQBUpkvDRqLWvG4pYaBjYHZFHqoJI5RxmwjRiL-VPlTkhZ6mKbT1kwdkISb0ZPCcDWy-s5DWoXYcECXv5EyhcUHbRqzGFGcsOjtgev6xq0YYP2U1CHhraNDlaLd5cf2C6TMDrXKWFkCItYImQA8YsIHefDzmQxpCyp3eUQ9E6fDYXU_4K_DoA_7p_pyGPvwqGTtgzjCylJcSkF6n4Z7zCLQ75XT-iN5axlB9IKRWsnyzssIJ649rtssrTXRh3ROtHpES8bVJrxq60fRNIWJGmerU7T9xzn71JqXjwXfVcUf4yd615N4O_P_eVEghmkBGOovYrrQ3IwRnoIc980yvRIWL0lBtp5Ly19FmD6V0gXLwCuLl2IBCmfzZqYCNL_mzVEX1LPkAGfu_4KbgaOZ_M536u3v6Z2Texd_Ru7xDUNNzq-tFac9MphgFQy0sP4kIsI-abAH2MGpE43Y8prYom5agmqu3xc-TqwQ1RJxZmkcsiVSSQtuEoIrPN-E2rboYvLq6dmshMNQeouooFvz6MBzJRbxFn8ydZxJamE2_4RrJzLiAiEwaL2aN8tNZUyJ-_Jms5c91DZb8ahPfYnS-RS-f1rpy-EGxKaD4XcVpymho3QUQF-KquS-Qu21ZpBAQcK37biYQoQstNbpXte5ePh2UhxU95CeHBKKMZ4DzxIOZhQdP4Sik_q_7GRYnF5ADsKtFhCVT07o7bFBCOnSgpR2LnGFVBvApl5b2V5jnVD-spOPaL8QPoYnkl-0aySyeJ4NXm36piXORsTXvU"  # ← paste the refresh token string here

if not REFRESH_TOKEN_VALUE:
    print("Paste the refresh token value from the cell above into REFRESH_TOKEN_VALUE and re-run.")
else:
    print(f"Saving refresh token to Key Vault secret '{PC_INCENTIVE_REFRESH_SECRET}'...")
    _kv_token = notebookutils.credentials.getToken("https://vault.azure.net")
    _kv_url   = f"{KEY_VAULT_URL.rstrip('/')}/secrets/{PC_INCENTIVE_REFRESH_SECRET}?api-version=7.4"
    _kv_resp  = requests.put(
        _kv_url,
        headers={"Authorization": f"Bearer {_kv_token}", "Content-Type": "application/json"},
        json={"value": REFRESH_TOKEN_VALUE},
        timeout=30,
    )
    _kv_resp.raise_for_status()
    print(f"✓ Saved to Key Vault as '{PC_INCENTIVE_REFRESH_SECRET}'")
    print()
    print("DONE — set RUN_CONSENT_SETUP = False in cell 5, then run Step 5 (MCI cell).")


StatementMeta(, 929c1108-0cb1-4e64-8d97-e2f9bfc54718, 31, Finished, Available, Finished, False)

Saving refresh token to Key Vault secret 'PCIncentiveRefreshToken'...
✓ Saved to Key Vault as 'PCIncentiveRefreshToken'

DONE — set RUN_CONSENT_SETUP = False in cell 5, then run Step 5 (MCI cell).


In [ ]:
# ============================================================================
# STEP 1 — Fetch all CSP customers  →  pc_customers
# ============================================================================
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType

print("Fetching customers from Partner Center...")
customer_rows = []

for c in pc_get_paged("/v1/customers"):
    profile = c.get("companyProfile") or {}
    billing = c.get("billingProfile") or {}
    address = billing.get("defaultAddress") or {}
    customer_rows.append({
        "customer_id"          : c.get("id"),
        "tenant_id"            : profile.get("tenantId"),
        "company_name"         : profile.get("companyName"),
        "domain"               : profile.get("domain"),
        "relationship"         : c.get("relationshipToPartner"),
        "country"              : address.get("country"),
        "city"                 : address.get("city"),
        "state"                : address.get("state"),
        "billing_email"        : billing.get("email"),
        "billing_contact_name" : (billing.get("firstName") or "") + " " + (billing.get("lastName") or ""),
        "allow_delegated_admin": c.get("allowDelegatedAccess"),
        "ingested_at"          : datetime.now(timezone.utc).isoformat(),
    })

customers_pdf = pd.DataFrame(customer_rows)
print(f"Fetched {len(customers_pdf)} customers.")

customers_sdf = spark.createDataFrame(customers_pdf)
# Explicitly cast columns that may be all-null to prevent VOID type in Delta
for _c in ["country", "city", "state", "billing_email", "allow_delegated_admin"]:
    customers_sdf = customers_sdf.withColumn(_c, F.col(_c).cast(StringType()))

(customers_sdf.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(f"{SCHEMA}.pc_customers"))
print(f"Wrote {SCHEMA}.pc_customers: {customers_sdf.count():,} rows")

# Save customer list for use in subsequent steps
customer_ids = customers_pdf[["customer_id", "company_name"]].to_dict("records")
customers_sdf.show(5, truncate=False)

StatementMeta(, 929c1108-0cb1-4e64-8d97-e2f9bfc54718, 33, Finished, Available, Finished, False)

Fetching customers from Partner Center...
Fetched 269 customers.
Wrote dbo.pc_customers: 269 rows
+------------------------------------+------------------------------------+-------------------------------------------------+-------------------------------------+------------+-------+----+-----+-------------+--------------------+---------------------+--------------------------------+
|customer_id                         |tenant_id                           |company_name                                     |domain                               |relationship|country|city|state|billing_email|billing_contact_name|allow_delegated_admin|ingested_at                     |
+------------------------------------+------------------------------------+-------------------------------------------------+-------------------------------------+------------+-------+----+-----+-------------+--------------------+---------------------+--------------------------------+
|92dc1ce1-8996-4e0b-b416-1c3ffeee5f1b|92dc1c

StatementMeta(, 6cc88787-a6db-45d9-b83f-f986c012fc45, 9, Finished, Available, Finished, False)

Fetching customers from Partner Center...
Fetched 269 customers.
Wrote dbo.pc_customers: 269 rows
+------------------------------------+------------------------------------+-------------------------------------------------+-------------------------------------+------------+-------+----+-----+-------------+--------------------+---------------------+--------------------------------+
|customer_id                         |tenant_id                           |company_name                                     |domain                               |relationship|country|city|state|billing_email|billing_contact_name|allow_delegated_admin|ingested_at                     |
+------------------------------------+------------------------------------+-------------------------------------------------+-------------------------------------+------------+-------+----+-----+-------------+--------------------+---------------------+--------------------------------+
|92dc1ce1-8996-4e0b-b416-1c3ffeee5f1b|92dc1c

In [16]:
# ============================================================================
# STEP 2 — Fetch subscriptions per customer  →  pc_subscriptions
# ============================================================================
print("Fetching subscriptions for all customers...")
sub_rows = []
customer_ids = customers_pdf[["customer_id", "company_name"]].dropna(subset=["customer_id"]).to_dict("records")

for i, cust in enumerate(customer_ids, 1):
    cid, name = cust["customer_id"], cust["company_name"]
    try:
        for sub in pc_get_paged(f"/v1/customers/{cid}/subscriptions"):
            sub_rows.append({
                "customer_id"               : cid,
                "company_name"              : name,
                "subscription_id"           : sub.get("id"),
                "offer_id"                  : sub.get("offerId"),
                "offer_name"                : sub.get("offerName"),
                "quantity"                  : sub.get("quantity"),
                "unit_type"                 : sub.get("unitType"),
                "billing_type"              : sub.get("billingType"),
                "billing_cycle"             : sub.get("billingCycle"),
                "effective_start_date"      : sub.get("effectiveStartDate"),
                "commitment_end_date"       : sub.get("commitmentEndDate"),
                "cancellation_allowed_until": sub.get("cancellationAllowedUntilDate"),
                "auto_renew_enabled"        : sub.get("autoRenewEnabled"),
                "status"                    : sub.get("status"),
                "is_trial"                  : sub.get("isTrial"),
                "ingested_at"               : datetime.now(timezone.utc).isoformat(),
            })
    except Exception as e:
        print(f"  [{i}/{len(customer_ids)}] WARN: {name} — {e}")
    if i % 10 == 0:
        print(f"  [{i}/{len(customer_ids)}] processed...")

subs_pdf = pd.DataFrame(sub_rows)
print(f"Total subscriptions collected: {len(subs_pdf)}")

subs_sdf = spark.createDataFrame(subs_pdf)
(subs_sdf.write.mode("overwrite").option("overwriteSchema", "true")
    .format("delta").saveAsTable(f"{SCHEMA}.pc_subscriptions"))
print(f"Wrote {SCHEMA}.pc_subscriptions: {subs_sdf.count():,} rows")
subs_sdf.groupBy("offer_name", "status").count().orderBy("count", ascending=False).show(20, truncate=False)

StatementMeta(, bcba2454-ceea-4860-b94b-3a8f15a81e7f, 29, Finished, Available, Finished, False)

Fetching subscriptions for all customers...
  [1/269] WARN: Lake Chad Peace and Development Initiative (LCPD) — 403 Client Error: Forbidden for url: https://api.partnercenter.microsoft.com/v1/customers/92dc1ce1-8996-4e0b-b416-1c3ffeee5f1b/subscriptions?size=300&offset=0
  [3/269] WARN: Akwa Ibom Investment Corporation (AKICORP) — 403 Client Error: Forbidden for url: https://api.partnercenter.microsoft.com/v1/customers/1eda022e-6c05-4cc4-9063-74dc611acee8/subscriptions?size=300&offset=0
  [4/269] WARN: Duo Fusion Enterprises Limited — 403 Client Error: Forbidden for url: https://api.partnercenter.microsoft.com/v1/customers/48d5fb19-2f47-4924-9086-76a8c2511fae/subscriptions?size=300&offset=0
  [5/269] WARN: Nigeria Inter-Bank Settlement System Plc — 403 Client Error: Forbidden for url: https://api.partnercenter.microsoft.com/v1/customers/c5cd8359-1172-457c-b3a6-38831701ec57/subscriptions?size=300&offset=0
  [6/269] WARN: Halkin Exploration and Production Limited — 403 Client Error: Forbi

In [ ]:
# ============================================================================
# STEP 3 — Fetch subscribed SKUs per customer  →  pc_subscribed_skus
# ============================================================================
print("Fetching subscribed SKUs for all customers...")
sku_rows = []

for i, cust in enumerate(customer_ids, 1):
    cid, name = cust["customer_id"], cust["company_name"]
    try:
        data = pc_get(f"/v1/customers/{cid}/subscribedskus")
        for sku in data.get("items", []):
            prepaid = sku.get("prepaidUnits") or {}
            sku_rows.append({
                "customer_id"       : cid,
                "company_name"      : name,
                "sku_id"            : sku.get("skuId"),
                "sku_part_number"   : sku.get("skuPartNumber"),
                "capability_status" : sku.get("capabilityStatus"),
                "consumed_units"    : sku.get("consumedUnits"),
                "prepaid_enabled"   : prepaid.get("enabled"),
                "prepaid_suspended" : prepaid.get("suspended"),
                "prepaid_warning"   : prepaid.get("warning"),
                "service_plans"     : json.dumps([
                    sp.get("servicePlanName") for sp in sku.get("servicePlans", [])
                    if sp.get("capabilityStatus") == "Enabled"
                ]),
                "ingested_at"       : datetime.now(timezone.utc).isoformat(),
            })
    except Exception as e:
        print(f"  [{i}/{len(customer_ids)}] WARN: {name} — {e}")
    if i % 10 == 0:
        print(f"  [{i}/{len(customer_ids)}] processed...")

print(f"Total SKU rows collected: {len(sku_rows)}")

if sku_rows:
    sku_sdf = spark.createDataFrame(pd.DataFrame(sku_rows))
    (sku_sdf.write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(f"{SCHEMA}.pc_subscribed_skus"))
    print(f"Wrote {SCHEMA}.pc_subscribed_skus: {sku_sdf.count():,} rows")
    sku_sdf.groupBy("sku_part_number", "capability_status").count().orderBy("count", ascending=False).show(20, truncate=False)
else:
    print("No SKU data retrieved — table not written.")
    print("Tip: Customers without DAP/GDAP relationships return 403 for subscribedskus.")

StatementMeta(, bcba2454-ceea-4860-b94b-3a8f15a81e7f, 30, Finished, Available, Finished, False)

Fetching subscribed SKUs for all customers...
  [1/269] WARN: Lake Chad Peace and Development Initiative (LCPD) — 403 Client Error: Forbidden for url: https://api.partnercenter.microsoft.com/v1/customers/92dc1ce1-8996-4e0b-b416-1c3ffeee5f1b/subscribedskus
  [2/269] WARN: Enviroserve Kenya Limited — 403 Client Error: Forbidden for url: https://api.partnercenter.microsoft.com/v1/customers/4bf7ee0d-5dd6-48cf-b56d-69c225e8f9fe/subscribedskus
  [3/269] WARN: Akwa Ibom Investment Corporation (AKICORP) — 403 Client Error: Forbidden for url: https://api.partnercenter.microsoft.com/v1/customers/1eda022e-6c05-4cc4-9063-74dc611acee8/subscribedskus
  [4/269] WARN: Duo Fusion Enterprises Limited — 403 Client Error: Forbidden for url: https://api.partnercenter.microsoft.com/v1/customers/48d5fb19-2f47-4924-9086-76a8c2511fae/subscribedskus
  [5/269] WARN: Nigeria Inter-Bank Settlement System Plc — 403 Client Error: Forbidden for url: https://api.partnercenter.microsoft.com/v1/customers/c5cd8359-1172-4

IndexError: list index out of range

In [7]:
# ============================================================================
# STEP 4 — Fetch active promotions  →  pc_promotions
# ============================================================================
# Grain: one row per promotion × required product (exploded).
# All nested fields (term, pricingPolicies, seat constraints) are flattened.
#
# Confirmed API response shape:
#   id, name, description, startDate, endDate
#   properties.isAutoApplicable
#   requiredProducts[]:
#     productId, skuId
#     term.duration, term.billingCycle
#     pricingPolicies[]: policyType, value
#   promotionConstraints.seatConstraints[]: minSeats, maxSeats
#   promotionConstraints.assetOwnershipLimits[]: maxAssets

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType, BooleanType

print(f"Fetching active promotions (country={PROMO_COUNTRY}, segment={PROMO_SEGMENT})...")
promo_rows = []

try:
    data = pc_get("/v1/productpromotions", params={"country": PROMO_COUNTRY, "segment": PROMO_SEGMENT})
    for p in data.get("items", []):
        props       = p.get("properties") or {}
        constraints = p.get("promotionConstraints") or {}
        seat_limits = (constraints.get("seatConstraints") or [{}])[0]
        redemptions = (constraints.get("assetOwnershipLimits") or [{}])[0]

        # Shared promotion-level fields
        promo_base = {
            "promotion_id"       : p.get("id"),
            "name"               : p.get("name"),
            "description"        : p.get("description"),
            "start_date"         : p.get("startDate"),
            "end_date"           : p.get("endDate"),
            "is_auto_applicable" : props.get("isAutoApplicable"),
            "min_seats"          : seat_limits.get("minSeats"),
            "max_seats"          : seat_limits.get("maxSeats"),
            "max_redemptions"    : redemptions.get("maxAssets"),
            "eligible_segments"  : json.dumps(p.get("applicableSegments", [])),
            "country"            : PROMO_COUNTRY,
            "segment"            : PROMO_SEGMENT,
            "ingested_at"        : datetime.now(timezone.utc).isoformat(),
        }

        # Explode: one row per required product
        req_products = p.get("requiredProducts") or []
        for prod in req_products:
            term    = prod.get("term") or {}
            pricing = (prod.get("pricingPolicies") or [{}])[0]
            raw_val = pricing.get("value")

            promo_rows.append({
                **promo_base,
                "product_id"          : prod.get("productId"),
                "sku_id"              : prod.get("skuId"),
                "term_duration"       : term.get("duration"),       # e.g. "P3Y"
                "billing_cycle"       : term.get("billingCycle"),   # e.g. "Triennial"
                "promotion_type"      : pricing.get("policyType"),  # e.g. "PercentDiscount"
                "discount_percentage" : float(raw_val) * 100 if raw_val is not None else None,
            })

    print(f"Found {len(promo_rows)} promotion × product rows from {len(data.get('items', []))} promotions.")
except Exception as e:
    print(f"WARN: Could not fetch promotions — {e}")

if promo_rows:
    promo_pdf = pd.DataFrame(promo_rows)
    promo_sdf = spark.createDataFrame(promo_pdf)

    # Explicit casts to avoid VOID for columns that may be all-null
    for _c in ["product_id", "sku_id", "promotion_type", "term_duration", "billing_cycle"]:
        promo_sdf = promo_sdf.withColumn(_c, F.col(_c).cast(StringType()))
    promo_sdf = promo_sdf.withColumn("discount_percentage", F.col("discount_percentage").cast(DoubleType()))
    promo_sdf = promo_sdf.withColumn("is_auto_applicable",  F.col("is_auto_applicable").cast(BooleanType()))

    (promo_sdf.write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(f"{SCHEMA}.pc_promotions"))
    print(f"Wrote {SCHEMA}.pc_promotions: {promo_sdf.count():,} rows")
    promo_sdf.select(
        "promotion_id", "name", "product_id", "sku_id",
        "promotion_type", "discount_percentage",
        "term_duration", "billing_cycle", "min_seats", "max_seats"
    ).show(10, truncate=False)
else:
    print("No promotions found for this market/segment — table not written.")

StatementMeta(, e606b887-c87d-427c-9b25-7ffa66eae19d, 12, Finished, Available, Finished, False)

Fetching active promotions (country=NG, segment=commercial)...
Found 373 promotion × product rows from 373 promotions.
Wrote dbo.pc_promotions: 373 rows
+------------------------------+-------------------------------------------------------------------------------------------------+------------+------+---------------+-------------------+-------------+-------------+---------+---------+
|promotion_id                  |name                                                                                             |product_id  |sku_id|promotion_type |discount_percentage|term_duration|billing_cycle|min_seats|max_seats|
+------------------------------+-------------------------------------------------------------------------------------------------+------------+------+---------------+-------------------+-------------+-------------+---------+---------+
|39NFJQT10GN2:0001:084R5MQ9QF2R|Microsoft 365 E7 (No Teams) - 3 year-ME7 CSP 3 Year 15% Off Promo                                |CFQ7TTBZZR6H

In [ ]:
# ============================================================================
# STEP 4b — Build PC catalog sku_id → Graph skuPartNumber mapping
#           →  pc_sku_catalog
# ============================================================================
# Background:
#   pc_promotions.sku_id  = Partner Center catalog SHORT code, e.g. "000R"
#   Graph subscribedSkus.skuPartNumber = technical name,  e.g. "Windows_365_Business_..."
#   These are two different ID systems — this step bridges them.
#
# Strategy (two complementary sources):
#   A. PC Catalog API: GET /v1/products/{productId}/skus/{skuId}
#      → sku title + any dynamicAttributes that expose a Graph-compatible ID
#   B. pc_subscriptions cross-join: extract product_id:sku_id from offer_id,
#      paired with offer_name — gives human-readable label per catalog combo
#
# Output: dbo.pc_sku_catalog
#   product_id | catalog_sku_id | sku_title | offer_name | sku_part_number (if resolvable)
# ============================================================================
import pandas as pd, json
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

print("Building PC catalog SKU → Graph skuPartNumber mapping...")
print()

# ── A. Catalog API: fetch SKU details for every promo product+sku ────────────
try:
    promos_df = spark.table(f"{SCHEMA}.pc_promotions") \
        .select("product_id", "sku_id") \
        .dropna(subset=["product_id", "sku_id"]) \
        .distinct() \
        .toPandas()
    print(f"Distinct product+sku combos in pc_promotions: {len(promos_df)}")
except Exception as e:
    print(f"WARN: could not load pc_promotions — {e}")
    promos_df = pd.DataFrame(columns=["product_id", "sku_id"])

catalog_rows = []
for _, row in promos_df.iterrows():
    pid, sid = row["product_id"], row["sku_id"]
    try:
        sku_data = pc_get(f"/v1/products/{pid}/skus/{sid}", params={"country": PROMO_COUNTRY})
        dyn      = sku_data.get("dynamicAttributes") or {}

        # Try multiple field names that may hold a Graph-compatible ID or part number
        resolved_sku_part = (
            dyn.get("skuPartNumber")    or
            dyn.get("SkuPartNumber")    or
            dyn.get("qualificationId")  or
            dyn.get("servicePlanId")    or
            None
        )

        catalog_rows.append({
            "product_id"            : pid,
            "catalog_sku_id"        : sid,
            "sku_title"             : sku_data.get("title"),
            "sku_description"       : sku_data.get("description"),
            "is_trial"              : sku_data.get("isTrial"),
            "sku_part_number"       : resolved_sku_part,   # populated if API exposes it
            "dynamic_attributes"    : json.dumps(dyn),     # raw dump for inspection
            "source"                : "catalog_api",
            "ingested_at"           : datetime.now(timezone.utc).isoformat(),
        })
        print(f"  ✓ {pid}:{sid}  →  \"{sku_data.get('title')}\"  sku_part_number={resolved_sku_part}")
    except Exception as e:
        print(f"  ✗ {pid}:{sid}  →  {e}")

# ── B. pc_subscriptions cross-reference: offer_id → offer_name ──────────────
# offer_id format: "{productId}:{catalogSkuId}:{availabilityId}"
# This gives human-readable offer_name for each catalog product+sku combo.
try:
    subs_map = (
        spark.table(f"{SCHEMA}.pc_subscriptions")
        .withColumn("product_id_x",      F.split(F.col("offer_id"), ":").getItem(0))
        .withColumn("catalog_sku_id_x",  F.split(F.col("offer_id"), ":").getItem(1))
        .select("product_id_x", "catalog_sku_id_x", "offer_name")
        .distinct()
        .toPandas()
    )
    print(f"\nOffer_name entries from pc_subscriptions: {len(subs_map)}")

    # Build lookup dict: (product_id, catalog_sku_id) → offer_name
    offer_name_map = {
        (r["product_id_x"], r["catalog_sku_id_x"]): r["offer_name"]
        for _, r in subs_map.iterrows()
    }

    # Patch offer_name into catalog_rows
    for row in catalog_rows:
        key = (row["product_id"], row["catalog_sku_id"])
        row["offer_name"] = offer_name_map.get(key)

    # Also add any subs combos NOT already in catalog_rows (products not in pc_promotions)
    existing = {(r["product_id"], r["catalog_sku_id"]) for r in catalog_rows}
    for _, row in subs_map.iterrows():
        key = (row["product_id_x"], row["catalog_sku_id_x"])
        if key not in existing:
            catalog_rows.append({
                "product_id"         : row["product_id_x"],
                "catalog_sku_id"     : row["catalog_sku_id_x"],
                "sku_title"          : None,
                "sku_description"    : None,
                "is_trial"           : None,
                "sku_part_number"    : None,
                "dynamic_attributes" : None,
                "offer_name"         : row["offer_name"],
                "source"             : "subscriptions_only",
                "ingested_at"        : datetime.now(timezone.utc).isoformat(),
            })
except Exception as e:
    print(f"WARN: could not load pc_subscriptions — {e}")
    for row in catalog_rows:
        row.setdefault("offer_name", None)

# ── Write pc_sku_catalog ─────────────────────────────────────────────────────
if catalog_rows:
    cat_pdf = pd.DataFrame(catalog_rows)
    cat_sdf = spark.createDataFrame(cat_pdf)
    for c in ["product_id", "catalog_sku_id", "sku_title", "sku_description",
              "sku_part_number", "dynamic_attributes", "offer_name", "source"]:
        cat_sdf = cat_sdf.withColumn(c, F.col(c).cast(StringType()))
    (cat_sdf.write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(f"{SCHEMA}.pc_sku_catalog"))
    print(f"\nWrote {SCHEMA}.pc_sku_catalog: {cat_sdf.count():,} rows")
    print()
    cat_sdf.select("product_id", "catalog_sku_id", "sku_title", "offer_name", "sku_part_number") \
        .show(30, truncate=False)

    # Inspect dynamic_attributes of promo SKUs to find any hidden Graph ID fields
    print("\n=== dynamic_attributes of promo SKUs (raw — check for skuPartNumber equivalents) ===")
    cat_sdf.filter(
        F.col("source") == "catalog_api"
    ).select("catalog_sku_id", "sku_title", "dynamic_attributes").show(20, truncate=False)
else:
    print("No catalog rows collected.")


StatementMeta(, de006c7b-faf7-4d54-acbc-f7566fd78903, 17, Finished, Available, Finished, False)

Building PC catalog SKU → Graph skuPartNumber mapping...

Distinct product+sku combos in pc_promotions: 135
  ✓ CFQ7TTC0LFLZ:001L  →  "Microsoft 365 E5 (no Teams)"  sku_part_number=None
  ✓ CFQ7TTC0MM8R:001P  →  "Microsoft 365 Copilot Business"  sku_part_number=None
  ✗ CFQ7TTC0LFLZ:0002  →  404 Client Error: Not Found for url: https://api.partnercenter.microsoft.com/v1/products/CFQ7TTC0LFLZ/skus/0002?country=NG
  ✗ CFQ7TTC0LFLZ:000Z  →  404 Client Error: Not Found for url: https://api.partnercenter.microsoft.com/v1/products/CFQ7TTC0LFLZ/skus/000Z?country=NG
  ✗ CFQ7TTC0LFLX:002M  →  404 Client Error: Not Found for url: https://api.partnercenter.microsoft.com/v1/products/CFQ7TTC0LFLX/skus/002M?country=NG
  ✓ CFQ7TTC0LFLX:0001  →  "Microsoft 365 E3"  sku_part_number=None
  ✓ CFQ7TTC0LFLX:0021  →  "Microsoft 365 E3 (no Teams)"  sku_part_number=None
  ✓ CFQ7TTC0LFLX:002P  →  "Microsoft 365 E3 - Unattended License - 3 year"  sku_part_number=None
  ✓ CFQ7TTC0LFLZ:0003  →  "Microsoft 365 E5 

In [ ]:
# ============================================================================
# STEP 4c — Patch pc_sku_catalog: compute sku_part_number from sku_title
# ============================================================================
# The PC catalog API does not return a Graph-compatible skuPartNumber.
# However, the sku_title maps cleanly to Graph skuPartNumber via:
#   1. Remove commas
#   2. Replace spaces with underscores
# e.g. "Microsoft 365 E5 (no Teams)" → "Microsoft_365_E5_(no_Teams)"
#      "Windows 365 Business 4 vCPU, 16 GB, 128 GB" → "Windows_365_Business_4_vCPU_16_GB_128_GB"

from pyspark.sql import functions as F

cat = spark.table(f"{SCHEMA}.pc_sku_catalog")

# Apply transformation: strip commas, collapse whitespace, replace spaces with _
cat = cat.withColumn(
    "sku_part_number",
    F.when(
        F.col("sku_title").isNotNull(),
        F.regexp_replace(
            F.regexp_replace(F.trim(F.col("sku_title")), r",", ""),
            r"\s+", "_"
        )
    ).otherwise(F.col("sku_part_number"))
)

(cat.write.mode("overwrite").option("overwriteSchema", "true")
    .format("delta").saveAsTable(f"{SCHEMA}.pc_sku_catalog"))

print(f"Updated pc_sku_catalog with computed sku_part_number: {cat.count():,} rows")
print()

# Show the promo SKUs with their computed sku_part_number
print("=== Promo SKUs → sku_part_number mapping ===")
promo_ids = spark.table(f"{SCHEMA}.pc_promotions").select("product_id", "sku_id").distinct()
(cat.join(promo_ids,
          (cat.product_id == promo_ids.product_id) & (cat.catalog_sku_id == promo_ids.sku_id),
          "inner")
    .select(cat.product_id, cat.catalog_sku_id, cat.sku_title, cat.sku_part_number, cat.offer_name)
    .distinct()
    .orderBy("sku_title")
    .show(30, truncate=False))

# Verify against known Cloudware SKUs — check overlap
print("=== Cross-check: Cloudware SKUs that match catalog entries ===")
graph_skus = spark.table(f"{SCHEMA}.graph_tenant_skus").filter(
    F.lower(F.col("short_name")).contains("cloudware")
).select("sku_part_number").distinct()

(graph_skus.join(
    cat.select("sku_part_number", "sku_title", "product_id", "catalog_sku_id").distinct(),
    "sku_part_number", "inner"
).show(20, truncate=False))


StatementMeta(, de006c7b-faf7-4d54-acbc-f7566fd78903, 18, Finished, Available, Finished, False)

Updated pc_sku_catalog with computed sku_part_number: 227 rows

=== Promo SKUs → sku_part_number mapping ===
+------------+--------------+---------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------+------------------------------------------------------------------+
|product_id  |catalog_sku_id|sku_title                                                                              |sku_part_number                                                                        |offer_name                                                        |
+------------+--------------+---------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------+------------------------------------------------------------------+
|CFQ7TTBZZR6G|0002          |Agent 365                                  

In [ ]:
# ============================================================================
# STEP 5 — MCI Engagement / Workshop Eligibility  →  pc_mci_engagements
# ============================================================================
# Reliance is enrolled in MCI. This cell:
#   1. Discovers all active MCI engagements Reliance can deliver
#   2. For each customer, queries which engagements they are eligible for
#   3. Writes one row per customer × engagement to dbo.pc_mci_engagements
#
# Uses pc_get_incentive() — delegated App+User token (Secure Application Model).
# Requires Key Vault secret "PCIncentiveRefreshToken".
# If not yet set up, run the "One-Time Consent Setup" cell first.
#
# Join to sem_dim_customer via tenant_id = pc_tenant_id.

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Step 1: Discover MCI engagements Reliance can deliver ───────────────────
print("Step 1: Discovering MCI engagements Reliance can deliver...")
engagements = []
_eng_error = None
for path in ["/v1/engagements", "/v1/incentives/engagements"]:
    try:
        resp  = pc_get_incentive(path)
        items = resp.get("items", resp.get("value", []))
        if items:
            engagements = items
            print(f"  ✓ Found {len(engagements)} engagements via {path}")
            break
        else:
            print(f"  {path} → 200 OK but no items returned")
            break
    except Exception as e:
        _eng_error = e
        status = str(e)[:100]
        if "PCIncentiveRefreshToken" in status or "refresh" in status.lower():
            print(f"  Refresh token not found in Key Vault — run 'One-Time Consent Setup' cell first.")
        elif "404" in status:
            print(f"  {path} → 404")
        elif "403" in status:
            print(f"  {path} → 403 (user may not have Incentive admin role)")
        else:
            print(f"  {path} → {status}")

if not engagements and _eng_error:
    print()
    print("─" * 60)
    print("Cannot retrieve MCI data. Ensure:")
    print("  1. 'One-Time Consent Setup' cell has been run")
    print(f"  2. Key Vault secret '{PC_INCENTIVE_REFRESH_SECRET}' exists")
    print("  3. The consenting user has 'Incentive admin' role in Partner Center")
    print("  4. The app has 'user_impersonation' delegated permission on")
    print("     Microsoft Partner Center (fa3d9a0c-3fb0-42cc-9193-47c7ecd2edbd)")
    print("─" * 60)

# ── Step 2: For each customer, query which engagements they qualify for ──────
print(f"\nStep 2: Checking per-customer eligibility for {len(customer_ids)} customers...")
mci_rows = []
errors_other = 0

for i, cust in enumerate(customer_ids, 1):
    cid  = cust["customer_id"]
    name = cust["company_name"]

    fetched = False
    for path in [
        f"/v1/customers/{cid}/engagementEligibilities",
        f"/v1/customers/{cid}/engagementeligibility",
    ]:
        try:
            resp  = pc_get_incentive(path)
            items = resp.get("items", resp.get("value", []))
            for item in items:
                mci_rows.append({
                    "customer_id"          : cid,
                    "tenant_id"            : cust.get("tenant_id", cid),
                    "company_name"         : name,
                    "engagement_id"        : item.get("engagementId") or item.get("id"),
                    "engagement_name"      : item.get("engagementName") or item.get("name"),
                    "solution_area"        : item.get("solutionArea"),
                    "partner_role"         : item.get("partnerRole"),
                    "eligibility_status"   : item.get("eligibilityStatus") or item.get("status"),
                    "ineligibility_reason" : item.get("ineligibilityReason"),
                    "enrollment_status"    : item.get("enrollmentStatus"),
                    "last_updated"         : item.get("lastModifiedDate") or item.get("lastUpdatedDate"),
                    "ingested_at"          : datetime.now(timezone.utc).isoformat(),
                })
            fetched = True
            break
        except Exception as e:
            if "404" not in str(e) and "400" not in str(e):
                errors_other += 1
                if errors_other <= 3:
                    print(f"  [{i}] WARN {name}: {str(e)[:80]}")
            break

    # Fallback: per-engagement queries if bulk endpoint unavailable
    if not fetched and engagements:
        for eng in engagements:
            eng_id   = eng.get("id") or eng.get("engagementId")
            eng_name = eng.get("name") or eng.get("engagementName")
            for path in [
                f"/v1/engagements/{eng_id}/customers/{cid}",
                f"/v1/incentives/engagements/{eng_id}/customers/{cid}",
            ]:
                try:
                    r = pc_get_incentive(path)
                    mci_rows.append({
                        "customer_id"          : cid,
                        "tenant_id"            : cust.get("tenant_id", cid),
                        "company_name"         : name,
                        "engagement_id"        : eng_id,
                        "engagement_name"      : eng_name,
                        "solution_area"        : eng.get("solutionArea"),
                        "partner_role"         : eng.get("partnerRole"),
                        "eligibility_status"   : r.get("eligibilityStatus") or r.get("status"),
                        "ineligibility_reason" : r.get("ineligibilityReason"),
                        "enrollment_status"    : r.get("enrollmentStatus"),
                        "last_updated"         : r.get("lastModifiedDate"),
                        "ingested_at"          : datetime.now(timezone.utc).isoformat(),
                    })
                    break
                except Exception:
                    pass

    if i % 25 == 0:
        print(f"  [{i}/{len(customer_ids)}] processed  —  {len(mci_rows)} rows so far")

print(f"\nTotal MCI rows collected : {len(mci_rows)}")

# ── Step 3: Write to Delta ───────────────────────────────────────────────────
if mci_rows:
    mci_sdf = spark.createDataFrame(pd.DataFrame(mci_rows))
    for _c in ["customer_id", "tenant_id", "company_name", "engagement_id",
               "engagement_name", "solution_area", "partner_role",
               "eligibility_status", "ineligibility_reason", "enrollment_status", "last_updated"]:
        mci_sdf = mci_sdf.withColumn(_c, F.col(_c).cast(StringType()))
    (mci_sdf.write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(f"{SCHEMA}.pc_mci_engagements"))
    print(f"Wrote {SCHEMA}.pc_mci_engagements: {mci_sdf.count():,} rows")
    mci_sdf.groupBy("engagement_name", "eligibility_status").count().orderBy(
        "engagement_name", "eligibility_status"
    ).show(50, truncate=False)
else:
    print("No MCI data written.")
    print("Complete the 'One-Time Consent Setup' cell to enable Incentives API access.")

StatementMeta(, 929c1108-0cb1-4e64-8d97-e2f9bfc54718, 34, Finished, Available, Finished, False)

Step 1: Discovering MCI engagements Reliance can deliver...
  /v1/engagements → 404
  /v1/incentives/engagements → 404

────────────────────────────────────────────────────────────
Cannot retrieve MCI data. Ensure:
  1. 'One-Time Consent Setup' cell has been run
  2. Key Vault secret 'PCIncentiveRefreshToken' exists
  3. The consenting user has 'Incentive admin' role in Partner Center
  4. The app has 'user_impersonation' delegated permission on
     Microsoft Partner Center (fa3d9a0c-3fb0-42cc-9193-47c7ecd2edbd)
────────────────────────────────────────────────────────────

Step 2: Checking per-customer eligibility for 269 customers...
  [25/269] processed  —  0 rows so far
  [50/269] processed  —  0 rows so far
  [75/269] processed  —  0 rows so far
  [100/269] processed  —  0 rows so far
  [125/269] processed  —  0 rows so far
  [150/269] processed  —  0 rows so far
  [175/269] processed  —  0 rows so far
  [200/269] processed  —  0 rows so far
  [225/269] processed  —  0 rows so far

In [ ]:
# ============================================================================
# DIAGNOSTIC — Incentive API endpoint probe
# ============================================================================
# Probes a broad set of Partner Center Incentive / MCI endpoints and prints
# the raw HTTP status + response body for each, so we can identify what's
# available and why certain paths 404.

import requests as _req

def _probe(label, path, use_incentive_token=True):
    token_fn = get_pc_incentive_token if use_incentive_token else get_pc_token
    headers = {
        "Authorization": f"Bearer {token_fn()}",
        "Accept": "application/json",
        "MS-Contract-Version": "v1",
        "X-Locale": "en-US",
    }
    url = f"{PC_BASE_URL}/{path.lstrip('/')}"
    try:
        r = _req.get(url, headers=headers, timeout=30)
        body = r.text[:300].replace("\n", " ")
        print(f"  [{r.status_code}] {label}")
        if r.status_code not in (200, 201):
            print(f"         → {body}")
        else:
            try:
                j = r.json()
                count = len(j.get("items", j.get("value", [])))
                print(f"         → {count} items  |  keys: {list(j.keys())[:6]}")
            except Exception:
                print(f"         → {body}")
    except Exception as e:
        print(f"  [ERR] {label}: {e}")

print("=" * 65)
print("Incentive / MCI endpoint probe  (delegated token)")
print("=" * 65)

# ── Partner-level engagement endpoints ──────────────────────────────────────
_probe("GET /v1/engagements",                    "/v1/engagements")
_probe("GET /v1/incentives/engagements",         "/v1/incentives/engagements")
_probe("GET /v1/engagements?type=MCI",           "/v1/engagements?type=MCI")
_probe("GET /v1/commerceincentives/engagements", "/v1/commerceincentives/engagements")

# ── MCI-specific paths seen in some Partner Center SDK samples ───────────────
_probe("GET /v1/mci/engagements",                "/v1/mci/engagements")
_probe("GET /v1/programs",                       "/v1/programs")
_probe("GET /v1/incentives/programs",            "/v1/incentives/programs")
_probe("GET /v1/incentives/summary",             "/v1/incentives/summary")
_probe("GET /v1/incentives/enrollments",         "/v1/incentives/enrollments")

# ── Per-partner summary ───────────────────────────────────────────────────────
_probe("GET /v1/engagements (app-only)",         "/v1/engagements", use_incentive_token=False)

# ── Sample per-customer probe (first customer) ───────────────────────────────
if customer_ids:
    cid = customer_ids[0]["customer_id"]
    name = customer_ids[0]["company_name"]
    print(f"\n  --- Per-customer probe: {name} ({cid}) ---")
    _probe(f"GET /v1/customers/{{cid}}/engagementEligibilities",
           f"/v1/customers/{cid}/engagementEligibilities")
    _probe(f"GET /v1/customers/{{cid}}/engagementeligibility",
           f"/v1/customers/{cid}/engagementeligibility")
    _probe(f"GET /v1/customers/{{cid}}/engagements",
           f"/v1/customers/{cid}/engagements")
    _probe(f"GET /v1/customers/{{cid}}/incentives/enrollments",
           f"/v1/customers/{cid}/incentives/enrollments")

print("\nDone. Share the output above to identify which endpoints are accessible.")


StatementMeta(, 6cc88787-a6db-45d9-b83f-f986c012fc45, 10, Finished, Available, Finished, False)

Incentive / MCI endpoint probe  (delegated token)
  [404] GET /v1/engagements
         → { "statusCode": 404, "message": "Resource not found" }
  [404] GET /v1/incentives/engagements
         → { "statusCode": 404, "message": "Resource not found" }
  [404] GET /v1/engagements?type=MCI
         → { "statusCode": 404, "message": "Resource not found" }
  [404] GET /v1/commerceincentives/engagements
         → { "statusCode": 404, "message": "Resource not found" }
  [404] GET /v1/mci/engagements
         → { "statusCode": 404, "message": "Resource not found" }
  [404] GET /v1/programs
         → { "statusCode": 404, "message": "Resource not found" }
  [404] GET /v1/incentives/programs
         → { "statusCode": 404, "message": "Resource not found" }
  [404] GET /v1/incentives/summary
         → { "statusCode": 404, "message": "Resource not found" }
  [404] GET /v1/incentives/enrollments
         → { "statusCode": 404, "message": "Resource not found" }
  [404] GET /v1/engagements (app-only)

In [ ]:
# ============================================================================
# UTILITY — Eligible workshops for a specific tenant ID
# ============================================================================
# Set LOOKUP_TENANT_ID to any tenant GUID (from pc_customers.tenant_id or
# sem_dim_customer.pc_tenant_id) to see which MCI workshops and engagements
# that tenant qualifies for.
#
# Can be run independently at any time — does not re-run the full ingestion.

LOOKUP_TENANT_ID = ""   # ← paste a tenant_id here, e.g. "4bf7ee0d-5dd6-48cf-b56d-69c225e8f9fe"

if not LOOKUP_TENANT_ID:
    print("Set LOOKUP_TENANT_ID to a tenant GUID and re-run this cell.")
else:
    print(f"Querying MCI engagement eligibility for tenant: {LOOKUP_TENANT_ID}\n")
    results = []

    # Resolve Partner Center customer_id from the tenant_id
    # (in most cases customer_id == tenant_id in the Partner Center data model)
    pc_customer_id = LOOKUP_TENANT_ID
    company_name   = "Unknown"
    try:
        cust_resp  = pc_get(f"/v1/customers/{LOOKUP_TENANT_ID}")
        pc_customer_id = cust_resp.get("id", LOOKUP_TENANT_ID)
        company_name   = (cust_resp.get("companyProfile") or {}).get("companyName", "Unknown")
        print(f"Customer : {company_name}")
        print(f"PC ID    : {pc_customer_id}\n")
    except Exception as e:
        print(f"Could not resolve customer record ({e}), querying with tenant ID directly.\n")

    # ── Option A: bulk eligibility endpoint ─────────────────────────────────
    fetched = False
    for path in [
        f"/v1/customers/{pc_customer_id}/engagementEligibilities",
        f"/v1/customers/{pc_customer_id}/engagementeligibility",
    ]:
        try:
            resp  = pc_get(path)
            items = resp.get("items", resp.get("value", []))
            if items:
                for item in items:
                    results.append({
                        "engagement_name"    : item.get("engagementName") or item.get("name"),
                        "solution_area"      : item.get("solutionArea", ""),
                        "partner_role"       : item.get("partnerRole", ""),
                        "eligibility_status" : item.get("eligibilityStatus") or item.get("status", ""),
                        "reason"             : item.get("ineligibilityReason") or "",
                        "enrollment_status"  : item.get("enrollmentStatus") or "",
                    })
                fetched = True
                print(f"Fetched {len(items)} engagement eligibilities via {path}")
                break
        except Exception as e:
            pass

    # ── Option B: per-engagement fallback using the engagements list ────────
    if not fetched:
        if not engagements:
            print("'engagements' variable not loaded — run cell 9 (Step 5) first to populate it.")
        else:
            print(f"Bulk endpoint unavailable — checking {len(engagements)} engagements individually...")
            for eng in engagements:
                eng_id   = eng.get("id") or eng.get("engagementId")
                eng_name = eng.get("name") or eng.get("engagementName")
                for path in [
                    f"/v1/engagements/{eng_id}/customers/{pc_customer_id}",
                    f"/v1/incentives/engagements/{eng_id}/customers/{pc_customer_id}",
                ]:
                    try:
                        r = pc_get(path)
                        results.append({
                            "engagement_name"    : eng_name,
                            "solution_area"      : eng.get("solutionArea", ""),
                            "partner_role"       : eng.get("partnerRole", ""),
                            "eligibility_status" : r.get("eligibilityStatus") or r.get("status", ""),
                            "reason"             : r.get("ineligibilityReason") or "",
                            "enrollment_status"  : r.get("enrollmentStatus") or "",
                        })
                        break
                    except Exception:
                        pass

    # ── Display results ──────────────────────────────────────────────────────
    if results:
        import pandas as pd
        df = pd.DataFrame(results)
        eligible   = df[df["eligibility_status"].str.lower() == "eligible"]
        ineligible = df[df["eligibility_status"].str.lower() != "eligible"]

        print(f"\n{'='*70}")
        print(f"MCI Engagement Eligibility — {company_name}")
        print(f"{'='*70}")
        print(f"\n✅ ELIGIBLE ({len(eligible)} engagements):")
        if len(eligible):
            print(eligible[["engagement_name", "solution_area", "partner_role",
                             "enrollment_status"]].to_string(index=False))
        else:
            print("  None")

        print(f"\n❌ NOT ELIGIBLE / OTHER ({len(ineligible)} engagements):")
        if len(ineligible):
            print(ineligible[["engagement_name", "eligibility_status",
                               "reason"]].to_string(index=False))
    else:
        print("No eligibility data returned. Verify the tenant is an active CSP customer.")
        print("Also ensure the app has 'Incentive admin' or 'Incentive user' role in Partner Center.")

In [20]:
# ============================================================================
# SUMMARY
# ============================================================================
print("=" * 70)
print("PARTNER CENTER INGESTION COMPLETE")
print("=" * 70)

for t in ["pc_customers", "pc_subscriptions", "pc_subscribed_skus",
          "pc_promotions", "pc_mci_engagements"]:
    try:
        cnt = spark.table(f"{SCHEMA}.{t}").count()
        print(f"  {SCHEMA}.{t:<30s} {cnt:>6,} rows")
    except Exception:
        print(f"  {SCHEMA}.{t:<30s} (not created)")

print()
print("Next: run build_semantic_layer to join pc_customers into sem_dim_customer")
print("      matching on tenant_id / domain.")

StatementMeta(, bcba2454-ceea-4860-b94b-3a8f15a81e7f, 33, Finished, Available, Finished, False)

PARTNER CENTER INGESTION COMPLETE
  dbo.pc_customers                      269 rows
  dbo.pc_subscriptions                  448 rows
  dbo.pc_subscribed_skus             (not created)
  dbo.pc_promotions                     373 rows
  dbo.pc_mci_engagements             (not created)

Next: run build_semantic_layer to join pc_customers into sem_dim_customer
      matching on tenant_id / domain.
